# GodelAI-Lite: Memory & Identity Continuity for Gemma 4
### Gemma 4 Good Hackathon -- Technical Submission

> **Core Thesis:** Small models fail not from lack of intelligence, but from lack of memory.
> GodelAI-Lite adds three lightweight augmentation layers -- zero fine-tuning required.

---

### Two-Layer Memory Architecture

| Layer | Framework | Scope | Active When |
|-------|-----------|-------|-------------|
| **Weight-level** | GodelAI (main) | Prevents catastrophic forgetting via EWC + Fisher Scaling | During training |
| **Context-level** | GodelAI-Lite (this) | Preserves conversation state via MemPalace + MACP + GIFP | During inference |

Both layers together form a complete memory system. This notebook demonstrates the **inference-time layer**.

*"Intelligence can scale through memory, not just parameters."*

## 1. Setup & Installation

In [ ]:
import subprocess, sys

def _install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Upgrade only transformers stack -- do NOT touch numpy/scipy/sklearn
# (upgrading numpy breaks scipy which breaks sklearn on Kaggle)
_install('--upgrade', 'transformers', 'accelerate', 'huggingface_hub')

# bitsandbytes: GPU quantization, best-effort
try:
    _install('bitsandbytes')
except Exception:
    pass

import transformers
print(f'Setup complete. transformers=={transformers.__version__}')

In [ ]:
import os, torch, json, math, time, re, glob as _glob
import numpy as np
from typing import Dict, List
from dataclasses import dataclass, asdict
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings; warnings.filterwarnings('ignore')

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')


## 2. Accelerator Detection
> v2.4 supports TPU v5e-8 (primary), GPU T4 (fallback), and CPU (last resort).

In [ ]:
# Detect and configure the best available accelerator
IS_TPU = os.path.exists('/dev/accel0') or 'TPU_NAME' in os.environ or 'COLAB_TPU_ADDR' in os.environ
IS_GPU = torch.cuda.is_available()

DEVICE = None
DTYPE  = torch.float32

if IS_TPU:
    try:
        import torch_xla.core.xla_model as xm
        DEVICE = xm.xla_device()
        DTYPE  = torch.bfloat16   # TPU native dtype -- always bfloat16
        print(f'Accelerator : TPU')
        print(f'Device      : {DEVICE}')
        print(f'dtype       : bfloat16')
    except ImportError:
        print('torch_xla not found -- falling back.')
        IS_TPU = False

if not IS_TPU and IS_GPU:
    DEVICE = torch.device('cuda')
    DTYPE  = torch.float16
    print(f'Accelerator : GPU -- {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'dtype       : float16')

if DEVICE is None:
    DEVICE = torch.device('cpu')
    DTYPE  = torch.float32
    print('Accelerator : CPU (no GPU/TPU found)')

print(f'\nFinal device: {DEVICE} | dtype: {DTYPE}')


## 3. MemPalace-Lite v2
> Temporal decay, pattern-based extraction, disk persistence, deduplication.

In [ ]:
@dataclass
class MemoryEntry:
    content: str
    timestamp: int
    relevance_score: float = 1.0
    category: str = 'general'


class MemPalaceLite:
    """Structured external memory for SLMs -- v2 with decay, persistence, smarter extraction."""

    def __init__(self, max_history: int = 10, max_facts: int = 20, decay_rate: float = 0.05):
        self.history: List[MemoryEntry] = []
        self.key_facts: List[MemoryEntry] = []
        self.patterns: List[str] = []
        self.max_history = max_history
        self.max_facts = max_facts
        self.decay_rate = decay_rate
        self.step_counter = 0

    def _decayed_relevance(self, entry: MemoryEntry) -> float:
        age = self.step_counter - entry.timestamp
        return entry.relevance_score * math.exp(-self.decay_rate * age)

    def add_to_history(self, interaction: str, category: str = 'interaction'):
        self.step_counter += 1
        self.history.append(MemoryEntry(interaction, self.step_counter, category=category))
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]

    def add_fact(self, fact: str, relevance: float = 1.0):
        fact = fact.strip()
        if not fact or len(fact) < 8:
            return
        for e in self.key_facts:
            if fact.lower()[:40] in e.content.lower() or e.content.lower()[:40] in fact.lower():
                return
        self.key_facts.append(MemoryEntry(fact, self.step_counter, relevance_score=relevance, category='fact'))
        if len(self.key_facts) > self.max_facts:
            self.key_facts.sort(key=self._decayed_relevance, reverse=True)
            self.key_facts = self.key_facts[:self.max_facts]

    def add_pattern(self, pattern: str):
        if pattern and pattern not in self.patterns:
            self.patterns.append(pattern)

    def get_context(self, top_facts: int = 5, top_history: int = 3) -> str:
        parts = []
        if self.key_facts:
            sorted_facts = sorted(self.key_facts, key=self._decayed_relevance, reverse=True)
            facts_text = '\n'.join(f'  * {f.content}' for f in sorted_facts[:top_facts])
            parts.append(f'[REMEMBERED FACTS]\n{facts_text}')
        if self.history:
            hist_text = '\n'.join(f'  {h.content}' for h in self.history[-top_history:])
            parts.append(f'[RECENT CONVERSATION]\n{hist_text}')
        if self.patterns:
            pat_text = '\n'.join(f'  -> {p}' for p in self.patterns[-2:])
            parts.append(f'[REASONING PATTERNS]\n{pat_text}')
        return '\n\n'.join(parts)

    def extract_facts(self, text: str, user_input: str = '') -> List[str]:
        facts = []
        combined = (user_input + ' ' + text).strip()
        sentences = [s.strip() for s in re.split(r'[.!?]', combined) if s.strip()]
        patterns = [
            r'my name is ([\w ]+)',
            r'i am (?:a |an )?([\w ]+)',
            r'i work (?:as|at|for) ([\w ]+)',
            r"i(?:'m| am) (?:currently )?(?:studying|working on|researching|building) ([\w ]+)",
            r'i (?:live|am based) (?:in|at) ([\w ,]+)',
        ]
        for sent in sentences:
            for p in patterns:
                if re.search(p, sent.lower()) and 8 < len(sent) < 200:
                    facts.append(sent[:200])
                    break
        if len(facts) < 3:
            for sent in sentences:
                if 20 < len(sent) < 150:
                    if any(kw in sent.lower() for kw in [' is ', ' are ', ' was ', 'capital', 'located', 'known for']):
                        if sent not in facts:
                            facts.append(sent[:150])
        seen, unique = set(), []
        for f in facts:
            k = f.lower()[:30]
            if k not in seen:
                seen.add(k); unique.append(f)
        return unique[:4]

    def save(self, path: str):
        data = {'history': [asdict(h) for h in self.history],
                'key_facts': [asdict(f) for f in self.key_facts],
                'patterns': self.patterns, 'step_counter': self.step_counter}
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2)
        print(f'Memory saved -> {path}  ({len(self.key_facts)} facts, {len(self.history)} history)')

    @classmethod
    def load(cls, path: str, **kwargs) -> 'MemPalaceLite':
        inst = cls(**kwargs)
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        inst.history = [MemoryEntry(**h) for h in data['history']]
        inst.key_facts = [MemoryEntry(**f) for f in data['key_facts']]
        inst.patterns = data['patterns']
        inst.step_counter = data.get('step_counter', 0)
        print(f'Memory loaded <- {path}  ({len(inst.key_facts)} facts, {len(inst.history)} history)')
        return inst

    def to_dict(self) -> Dict:
        return {'history': [asdict(h) for h in self.history],
                'key_facts': [asdict(f) for f in self.key_facts],
                'patterns': self.patterns}


## 4. MACP-Lite
> Multi-step reasoning continuity across inference turns.

In [ ]:
@dataclass
class ReasoningStep:
    step_id: int
    input_context: str
    model_output: str
    confidence: float
    next_action: str


class MACPLite:
    """Multi-Agent Continuity Protocol (Lite): structured reasoning chain."""

    def __init__(self):
        self.reasoning_chain: List[ReasoningStep] = []
        self.current_step = 0
        self.context_buffer = ''

    def start_chain(self, initial_input: str):
        self.context_buffer = initial_input

    def add_step(self, input_ctx: str, output: str, confidence: float, next_action: str):
        self.reasoning_chain.append(ReasoningStep(
            self.current_step, input_ctx[:200], output[:300], confidence, next_action))
        self.current_step += 1
        self.context_buffer = output

    def get_chain_summary(self) -> str:
        if not self.reasoning_chain:
            return 'No reasoning chain yet.'
        lines = ['[REASONING CHAIN]']
        for s in self.reasoning_chain[-5:]:
            lines.append(f'  Step {s.step_id} (conf={s.confidence:.2f}): {s.model_output[:80]}...')
        return '\n'.join(lines)

    def reset(self):
        self.reasoning_chain = []; self.current_step = 0; self.context_buffer = ''


## 5. GIFP-Lite v2
> TF-IDF cosine consistency + hard contradiction penalty.

In [ ]:
class GIFPLite:
    """GIFP-Lite v2: TF-IDF cosine consistency + hard contradiction penalty."""

    def __init__(self, role: str = 'helpful assistant with persistent memory'):
        self.role_definition = role
        self.constraints: List[str] = []
        self.behavior_history: List[str] = []

    def set_constraints(self, constraints: List[str]):
        self.constraints = constraints

    def get_identity_prompt(self) -> str:
        prompt = f'You are a {self.role_definition}.\n'
        if self.constraints:
            prompt += '\nBehavioural guidelines:\n'
            for c in self.constraints:
                prompt += f'- {c}\n'
        prompt += '\nMaintain consistency across all interactions.\n'
        return prompt

    def check_consistency(self, output: str) -> float:
        hard = ['never mind', 'scratch that', 'i was wrong', 'i made an error']
        penalty = sum(0.15 for phrase in hard if phrase in output.lower())
        if len(self.behavior_history) < 3:
            return max(0.2, 1.0 - penalty)
        try:
            corpus = self.behavior_history[-6:] + [output]
            vec = TfidfVectorizer(stop_words='english', max_features=300)
            tfidf = vec.fit_transform(corpus)
            sims = cosine_similarity(tfidf[-1:], tfidf[:-1])[0]
            scaled = min(1.0, float(np.mean(sims)) * 2.0 + 0.3)
            return max(0.1, scaled - penalty)
        except Exception:
            return max(0.2, 0.75 - penalty)

    def record_behavior(self, output: str):
        if output and len(output) > 10:
            self.behavior_history.append(output[:300])
        if len(self.behavior_history) > 20:
            self.behavior_history = self.behavior_history[-20:]


## 6. Systems -- GodelAI-Lite & Baseline
> Both share the **same Gemma 4 weights**. Only the augmentation layer differs.
> This is the only honest way to measure the value added by the memory architecture.

In [ ]:
class GodelAILite:
    """Gemma 4 + MemPalace-Lite v2 + MACP-Lite + GIFP-Lite v2."""

    def __init__(self, model, tokenizer, memory_path: str = None):
        self.model = model
        self.tokenizer = tokenizer
        if memory_path and os.path.exists(memory_path):
            self.memory = MemPalaceLite.load(memory_path)
        else:
            self.memory = MemPalaceLite()
        self.continuity = MACPLite()
        self.identity = GIFPLite(role='helpful AI assistant with persistent memory')
        self.identity.set_constraints([
            'Always be helpful and accurate',
            'Reference previous context when relevant',
            'Maintain logical consistency across turns',
            'Acknowledge uncertainty when present'
        ])

    def build_prompt(self, user_input: str) -> str:
        identity = self.identity.get_identity_prompt()
        ctx = self.memory.get_context()
        if ctx:
            return f'{identity}\n{ctx}\n\n[CURRENT INPUT]\nUser: {user_input}\n\nAssistant:'
        return f'{identity}\nUser: {user_input}\n\nAssistant:'

    def _generate(self, prompt: str, max_tokens: int = 256) -> str:
        inputs = self.tokenizer(prompt, return_tensors='pt',
                                truncation=True, max_length=1024)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        cfg = GenerationConfig(max_new_tokens=max_tokens, temperature=0.7, top_p=0.9,
                               do_sample=True, pad_token_id=self.tokenizer.eos_token_id)
        with torch.no_grad():
            out = self.model.generate(**inputs, generation_config=cfg)
        # Sync TPU before decode
        if IS_TPU:
            import torch_xla.core.xla_model as xm
            xm.mark_step()
        return self.tokenizer.decode(
            out[0, inputs['input_ids'].shape[1]:].cpu(),
            skip_special_tokens=True).strip()

    def chat(self, user_input: str, refine: bool = False) -> str:
        self.continuity.start_chain(user_input)
        response = self._generate(self.build_prompt(user_input))
        score = self.identity.check_consistency(response)
        self.continuity.add_step(user_input, response, score,
                                  'continue' if score > 0.7 else 'refine')
        for fact in self.memory.extract_facts(response, user_input=user_input):
            self.memory.add_fact(fact)
        self.memory.add_to_history(f'User: {user_input}', 'user_input')
        self.memory.add_to_history(f'Assistant: {response[:200]}', 'assistant_output')
        self.identity.record_behavior(response)
        if refine and score < 0.5:
            refined = self._generate(self.build_prompt(f'Provide a clearer answer to: {user_input}'))
            return f'{response}\n\n[Refined]: {refined}'
        return response

    def save_memory(self, path: str): self.memory.save(path)
    def load_memory(self, path: str): self.memory = MemPalaceLite.load(path)
    def get_memory_state(self) -> Dict: return self.memory.to_dict()
    def get_reasoning_chain(self) -> str: return self.continuity.get_chain_summary()


class BaselineGemma:
    """Plain Gemma 4 -- NO memory, NO identity layer, NO continuity. Honest baseline."""

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self._history: List[str] = []

    def chat(self, user_input: str) -> str:
        prompt = f'User: {user_input}\nAssistant:'
        inputs = self.tokenizer(prompt, return_tensors='pt',
                                truncation=True, max_length=512)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        cfg = GenerationConfig(max_new_tokens=256, temperature=0.7, top_p=0.9,
                               do_sample=True, pad_token_id=self.tokenizer.eos_token_id)
        with torch.no_grad():
            out = self.model.generate(**inputs, generation_config=cfg)
        if IS_TPU:
            import torch_xla.core.xla_model as xm
            xm.mark_step()
        response = self.tokenizer.decode(
            out[0, inputs['input_ids'].shape[1]:].cpu(),
            skip_special_tokens=True).strip()
        self._history.extend([f'User: {user_input}', f'Assistant: {response}'])
        return response

    def reset(self): self._history = []


## 7. Initialize -- Load Gemma 4 on TPU

In [ ]:
# Kaggle mounts model at one of these paths depending on how it was added
KAGGLE_PATH_CANDIDATES = [
    '/kaggle/input/gemma-4/transformers/gemma-4-e4b-it/1',
    '/kaggle/input/gemma/transformers/gemma-4-e4b-it/1',
    '/kaggle/input/gemma-4e4b-it/transformers/gemma-4-e4b-it/1',
]
KAGGLE_MODEL_PATH = next((p for p in KAGGLE_PATH_CANDIDATES if os.path.isdir(p)), None)

# HF candidates -- Gemma 2-2b-it is GATED (requires license), do not use
HF_CANDIDATES = [
    'google/gemma-4-E4B-it',
    'google/gemma-4-E2B-it',
]

CANDIDATES = []
if KAGGLE_MODEL_PATH:
    CANDIDATES.append(KAGGLE_MODEL_PATH)
    print(f'Kaggle model found: {KAGGLE_MODEL_PATH} -- offline, no rate limits')
else:
    print('Kaggle model path not found -- using HuggingFace download')
CANDIDATES.extend(HF_CANDIDATES)

model, tokenizer, MODEL_NAME = None, None, None
t0 = time.time()

for candidate in CANDIDATES:
    try:
        print(f'Loading {candidate}...')
        tokenizer = AutoTokenizer.from_pretrained(candidate)

        if IS_TPU:
            model = AutoModelForCausalLM.from_pretrained(
                candidate, torch_dtype=DTYPE, low_cpu_mem_usage=True)
            model = model.to(DEVICE)
        elif IS_GPU:
            load_kw = dict(torch_dtype=DTYPE, device_map='auto', low_cpu_mem_usage=True)
            try:
                from transformers import BitsAndBytesConfig
                load_kw['quantization_config'] = BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True)
            except Exception:
                pass
            model = AutoModelForCausalLM.from_pretrained(candidate, **load_kw)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                candidate, torch_dtype=DTYPE, low_cpu_mem_usage=True)

        model.eval()
        MODEL_NAME = candidate if candidate in HF_CANDIDATES else 'google/gemma-4-E4B-it'
        print(f'Loaded in {time.time()-t0:.1f}s  |  device: {next(model.parameters()).device}')
        break
    except Exception as e:
        print(f'  Failed: {e.__class__.__name__}: {str(e)[:120]}')
        model, tokenizer = None, None

if model is None:
    raise RuntimeError('All model candidates failed. Ensure internet is ON and Gemma 4 model input is added.')

print(f'Model      : {MODEL_NAME}')
print(f'Parameters : {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

godelai  = GodelAILite(model=model, tokenizer=tokenizer)
baseline = BaselineGemma(model=model, tokenizer=tokenizer)
print(f'GodelAI-Lite = {MODEL_NAME} + MemPalace-Lite v2 + MACP-Lite + GIFP-Lite v2')
print(f'Baseline     = {MODEL_NAME} only (zero augmentation)')

## 8. Live Demo -- Memory in Action

In [ ]:
demo = GodelAILite(model=model, tokenizer=tokenizer)

print('Turn 1 -- inject personal facts:')
r1 = demo.chat('My name is Alex and I am a marine biologist based in Hawaii.')
print(f'  {r1[:200]}\n')

print('Turn 2 -- distractor:')
r2 = demo.chat('What is the capital of France?')
print(f'  {r2[:200]}\n')

print('Turn 3 -- memory recall:')
r3 = demo.chat('What do you remember about me?')
print(f'  {r3[:300]}\n')

print('Turn 4 -- context-dependent question:')
r4 = demo.chat('Given my background, what ocean project would you recommend?')
print(f'  {r4[:300]}\n')

state = demo.get_memory_state()
print(f'Facts stored: {len(state["key_facts"])}')
for f in state['key_facts']:
    print(f'  * {f["content"][:80]}')


In [ ]:
MEMORY_PATH = '/kaggle/working/godelai_memory.json'
demo.save_memory(MEMORY_PATH)

demo2 = GodelAILite(model=model, tokenizer=tokenizer, memory_path=MEMORY_PATH)
r = demo2.chat('What was my name again?')
print(f'Restored agent: {r[:200]}')


## 9. Quantitative Evaluation -- GodelAI-Lite vs Baseline

Three standardised tests. **Same model weights. Only augmentation differs.**

| Test | What it measures |
|------|------------------|
| Memory Retention | 3 injected facts, 3 distractors, 3 recall queries |
| Response Consistency | Same question x 5, TF-IDF cosine similarity |
| Context Coherence | 3 context turns -> 3 dependent questions |

In [ ]:
class EvaluationSuite:

    def _reset(self, system):
        if isinstance(system, GodelAILite):
            system.memory = MemPalaceLite()
            system.identity.behavior_history = []
            system.continuity.reset()
        else:
            system.reset()

    def memory_retention(self, godelai, baseline) -> Dict:
        facts = [
            'My name is Jordan and I am a marine biologist.',
            'I work at the Pacific Ocean Research Institute in Hawaii.',
            'I am currently studying humpback whale migration patterns.'
        ]
        distractors = [
            'What is 15 multiplied by 7?',
            'Tell me something about ancient Rome.',
            'What is the speed of light?'
        ]
        recall_qs = [
            ('What is my name and profession?',     ['jordan', 'marine biolog']),
            ('Where do I work?',                    ['pacific', 'hawaii', 'research']),
            ('What am I currently researching?',    ['humpback', 'whale', 'migration'])
        ]
        results = {}
        for name, sys in [('GodelAI-Lite', godelai), ('Baseline', baseline)]:
            self._reset(sys)
            for f in facts:       sys.chat(f)
            for d in distractors: sys.chat(d)
            recalled = 0
            for q, kws in recall_qs:
                r = sys.chat(q).lower()
                hit = any(k in r for k in kws)
                recalled += int(hit)
                print(f'  [{name}] {q[:45]}: {"PASS" if hit else "FAIL"}')
            results[name] = recalled / len(recall_qs)
        return results

    def consistency(self, godelai, baseline) -> Dict:
        q = 'What is artificial intelligence and why does it matter?'
        results = {}
        for name, sys in [('GodelAI-Lite', godelai), ('Baseline', baseline)]:
            self._reset(sys)
            responses = [sys.chat(q) for _ in range(5)]
            try:
                tfidf = TfidfVectorizer(stop_words='english').fit_transform(responses)
                m = cosine_similarity(tfidf)
                n = len(responses)
                pairs = [m[i][j] for i in range(n) for j in range(i+1, n)]
                score = float(np.mean(pairs))
            except Exception:
                score = 0.0
            results[name] = score
            print(f'  [{name}] TF-IDF cosine avg: {score:.4f}')
        return results

    def context_coherence(self, godelai, baseline) -> Dict:
        ctx_turns = [
            'I love hiking. My favourite trail is the Appalachian Trail.',
            'I recently adopted a golden retriever puppy named Biscuit.',
            'I am planning a career change into data science.'
        ]
        test_qs = [
            ('What outdoor activities suit my lifestyle?',
             ['hik', 'trail', 'appalachian', 'outdoor', 'mountain']),
            ('What should I consider when naming my next pet?',
             ['biscuit', 'golden', 'retriever', 'dog', 'puppy']),
            ('What technical skills should I focus on first?',
             ['data science', 'python', 'career', 'data', 'machine learning'])
        ]
        results = {}
        for name, sys in [('GodelAI-Lite', godelai), ('Baseline', baseline)]:
            self._reset(sys)
            for t in ctx_turns: sys.chat(t)
            coherent = 0
            for q, kws in test_qs:
                r = sys.chat(q).lower()
                hit = any(k in r for k in kws)
                coherent += int(hit)
                print(f'  [{name}] {q[:45]}: {"PASS" if hit else "FAIL"}')
            results[name] = coherent / len(test_qs)
        return results

    def run_all(self, godelai, baseline) -> Dict:
        print('='*60 + '\nEVALUATION: GodelAI-Lite vs Baseline Gemma\n' + '='*60)
        print('\nTest 1/3 -- Memory Retention')
        t1 = self.memory_retention(godelai, baseline)
        print('\nTest 2/3 -- Response Consistency')
        t2 = self.consistency(godelai, baseline)
        print('\nTest 3/3 -- Context Coherence')
        t3 = self.context_coherence(godelai, baseline)
        return {'memory_retention': t1, 'consistency': t2, 'context_coherence': t3}


In [ ]:
results = EvaluationSuite().run_all(godelai, baseline)

METRICS = [
    ('memory_retention',  'Memory Retention    '),
    ('consistency',       'Response Consistency'),
    ('context_coherence', 'Context Coherence   '),
]

print('\n' + '='*65)
print('  GODELAI-LITE vs BASELINE GEMMA 4 -- RESULTS')
print('='*65)
print(f'{"Metric":<25} {"Baseline":>10} {"GodelAI-Lite":>14} {"Delta":>10}')
print('-'*65)
for key, label in METRICS:
    b = results[key]['Baseline']
    g = results[key]['GodelAI-Lite']
    delta = ((g - b) / b * 100) if b > 0 else float('inf')
    ds = f'+{delta:.1f}%' if delta >= 0 else f'{delta:.1f}%'
    print(f'{label:<25} {b:>10.3f} {g:>14.3f} {ds:>10}')
print('-'*65)
b_avg = np.mean([results[k]['Baseline'] for k, _ in METRICS])
g_avg = np.mean([results[k]['GodelAI-Lite'] for k, _ in METRICS])
od = ((g_avg - b_avg) / b_avg * 100) if b_avg > 0 else 0
print(f'{"OVERALL AVERAGE":<25} {b_avg:>10.3f} {g_avg:>14.3f} {"+"+str(round(od,1))+"%" if od>=0 else str(round(od,1))+"%":>10}')
print('='*65)


## 10. Architecture Summary & Ecosystem Connection

```
Training Time  ->  GodelAI (main)  : T-Score + EWC + Fisher Scaling
                                     82.8% catastrophic forgetting reduction
                                     Weight-level memory preservation

Inference Time ->  GodelAI-Lite    : MemPalace-Lite v2 + MACP-Lite + GIFP-Lite v2
                                     Zero fine-tuning required
                                     Context-level memory preservation
```

**Both layers together = complete memory system for continual AI.**

In [ ]:
print('='*60)
print('GODELAI-LITE v2.4 -- ARCHITECTURE SUMMARY')
print('='*60)
summary = '''
MemPalace-Lite v2   [Memory Layer]
  + Temporal decay (exponential relevance fade)
  + Pattern-based fact extraction
  + Disk persistence (JSON save/load)
  + Deduplication

MACP-Lite           [Continuity Layer]
  + Structured reasoning chain per turn
  + Confidence tracking

GIFP-Lite v2        [Identity Layer]
  + TF-IDF cosine consistency
  + Hard contradiction penalty

Base Model          Gemma 4  (zero fine-tuning)
Accelerator         TPU v5e-8 (bfloat16)  |  GPU T4 (float16)  |  CPU
Dependencies        torch, transformers, scikit-learn, numpy
'''
print(summary)
print('Ecosystem: GodelAI-Lite closes the inference-time memory gap.')
print('           GodelAI (main) closes the training-time memory gap.')
print('           Together: complete two-layer memory for continual AI.')


## References

- [GodelAI Framework](https://zenodo.org/records/18048374) -- Zenodo DOI: 10.5281/zenodo.18048374
- [Gemma Models](https://huggingface.co/google) -- Google Gemma model family
- [Elastic Weight Consolidation](https://arxiv.org/abs/1612.00796) -- Kirkpatrick et al., 2017

---
**Gemma 4 Good Hackathon -- Submission by creator35lwb**  
*"Intelligence can scale through memory, not just parameters."*